# Download DE Africa CCI Land Cover data for Ghana study area

This notebook downloads the ESA CCI Land Cover product (`cci_landcover`, 300 m resolution, annual 1992–2022) from Digital Earth Africa's public STAC catalog for a study area in Ghana, using `pystac-client` + `odc-stac`.

No Digital Earth Africa Sandbox account or local Open Data Cube is required — this pulls data directly from DE Africa's open S3 archive via STAC.

**Requirements:** see `requirements.txt` in the repo root. Package: `deafrica-tools` is used for plotting with the correct CCI legend.

## 1. Load packages

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import geopandas as gpd
from pystac_client import Client
from odc.stac import stac_load
from deafrica_tools.plotting import plot_lulc

ModuleNotFoundError: No module named 'matplotlib'

## 2. Define study area — Ankobra river basin

The study area is derived from the OSM waterways shapefile (`data/raw/shapefiles/osm_waterways/waterways_lines.shp`, EPSG:4326):

1. Keep **natural watercourses only** (same `NATURAL_WATERWAYS` filter as `d_03_waterways.R`).
2. Seed on all segments named *Ankobra* (12 mainstem segments), then grow the **connected river network** by iteratively adding every natural watercourse that touches the network — this pulls in the tributaries (e.g. the Bonsa; ~48 segments, ~590 km of network).
3. The basin proxy is the **convex hull of that network, buffered by 5 km**; its bounding box drives the STAC query.

Note this is a *network-hull proxy* for the true hydrological basin (OSM lines carry no watershed divides). It is wider than the mainstem-bbox used for the MERIT export (`merit_hydro_ankobra.tif`: −2.36 to −2.01°E, 4.81–5.82°N) because it includes the tributary network up to ~6.4°N.

In [ ]:
import numpy as np

# Natural waterway types (mirrors NATURAL_WATERWAYS in d_03_waterways.R)
NATURAL_WATERWAYS = ["river", "stream", "brook", "wadi", "tidal_channel",
                     "stream_pool", "flowline"]

waterways_path = "../../data/raw/shapefiles/osm_waterways/waterways_lines.shp"
ww = gpd.read_file(waterways_path)  # EPSG:4326
nat = ww[ww["waterway"].isin(NATURAL_WATERWAYS)].reset_index(drop=True)

# Seed: every segment named Ankobra (mainstem)
in_network = nat["name"].str.contains("Ankobra", case=False, na=False).to_numpy()
print(f"Natural watercourses: {len(nat)} | Ankobra seed segments: {in_network.sum()}")

# Grow the connected network: add any natural watercourse touching the
# current network until no new segment joins (picks up tributaries, e.g. Bonsa)
sidx = nat.sindex
frontier = np.where(in_network)[0]
while len(frontier):
    candidates = set()
    for i in frontier:
        candidates.update(sidx.query(nat.geometry.iloc[i], predicate="intersects"))
    frontier = np.array([c for c in candidates if not in_network[c]], dtype=int)
    in_network[frontier] = True

ankobra_net = nat[in_network]
net_km = ankobra_net.to_crs(32630).length.sum() / 1000
print(f"Connected Ankobra network: {in_network.sum()} segments, {net_km:,.0f} km")

# Basin proxy: convex hull of the network, buffered 5 km (buffer in UTM30N)
aoi = (
    gpd.GeoSeries([ankobra_net.union_all().convex_hull], crs=4326)
    .to_crs(32630).buffer(5_000).to_crs(4326)
)

bbox = list(aoi.total_bounds)  # [min_lon, min_lat, max_lon, max_lat]
print("Bounding box (min_lon, min_lat, max_lon, max_lat):", [round(v, 4) for v in bbox])

## 3. Set the year(s) of interest

`cci_landcover` is available annually from 1992 to 2022. Set one year, or a list of years to compare land cover change over time.

In [ ]:
years = [2020]  # e.g. [2000, 2010, 2020] to look at change over time

## 4. Search and load data from the DE Africa STAC catalog

In [ ]:
catalog = Client.open("https://explorer.digitalearth.africa/stac")

datasets = {}

for year in years:
    query = catalog.search(
        collections=["cci_landcover"],
        bbox=bbox,
        datetime=f"{year}-01-01/{year}-12-31",
    )
    items = list(query.items())
    print(f"{year}: found {len(items)} item(s)")

    ds = stac_load(
        items,
        bands=["classification"],
        bbox=bbox,
        resolution=0.0027,  # ~300 m in degrees
    )
    datasets[year] = ds.squeeze()

datasets[years[0]]

## 5. Plot the land cover map(s)

In [ ]:
for year, ds in datasets.items():
    fig, ax = plt.subplots(figsize=(10, 10))
    plot_lulc(ds.classification, product="CCI", legend=True, ax=ax)
    ax.set_title(f"CCI Land Cover — {year}")
    plt.tight_layout()
    plt.show()

## 6. Save data locally (optional)

Export each year's classification layer to GeoTIFF for use in QGIS or elsewhere.

In [ ]:
import os

import rioxarray  # noqa: F401 — registers the .rio accessor

output_dir = "../../data/raw/land_cover"
os.makedirs(output_dir, exist_ok=True)

for year, ds in datasets.items():
    out_path = os.path.join(output_dir, f"cci_landcover_ankobra_{year}.tif")
    ds.classification.rio.to_raster(out_path)
    print(f"Saved: {out_path}")